# Re-OCR via PERO OCR (moteur GPU DCGM/pero-ocr)

Pipeline de re-OCRisation utilisant [pero-ocr](https://github.com/DCGM/pero-ocr) (analyse de mise en page + reconnaissance de lignes) sur les images Gallica IIIF ou locales.

**Par rapport au pipeline Mistral (`reocr_mistral.ipynb`) :**
- Nécessite un GPU (Colab : `Exécution > Modifier le type d'exécution > GPU`)
- Détecte lui-même les lignes de texte (pas besoin des blocs ALTO existants comme entrée — mais on les garde pour comparaison)
- Sortie directement en ALTO XML, compatible avec `stats_pages.py` (source `"Pero"`)
- Plus lent par page mais un seul modèle local, pas d'appels API payants

**Avant de lancer :** il faut un modèle PERO OCR pré-entraîné (config + checkpoints).
Va sur le [model zoo officiel](https://pero-ocr.fit.vutbr.cz/models), choisis un modèle adapté
(idéalement un modèle "print"/imprimé, français ou multilingue européen) et récupère l'URL
du zip — à coller dans la cellule de configuration (`MODEL_ZIP_URL`).

In [ ]:
# ── Cellule 1 : Dépendances ────────────────────────────────────────────────
!pip install pero-ocr opencv-python-headless lxml requests -q

import torch
print('CUDA disponible :', torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠ Pas de GPU détecté — Exécution > Modifier le type d'exécution > GPU")


In [ ]:
# ── Cellule 2 : Montage Drive ──────────────────────────────────────────────
import os, shutil

mountpoint = '/content/gdrive'   # on évite /content/drive qui pose problème
if os.path.isdir(mountpoint):
    os.system(f'fusermount -uz {mountpoint} 2>/dev/null || umount -l {mountpoint} 2>/dev/null')
    shutil.rmtree(mountpoint, ignore_errors=True)

from google.colab import drive
drive.mount(mountpoint)
print('Drive monté sur', mountpoint)


In [ ]:
# ── Cellule 3 : Configuration ──────────────────────────────────────────────

# Dossier source : METS/ALTO originaux BnF (identique au pipeline Mistral)
# Structure attendue : SOURCE_DIR/<fascicule_id>_reocr/{toc/T*.xml, ocr/X*.xml, manifest.xml}
SOURCE_DIR = '/content/gdrive/MyDrive/CHEMIN/VERS/TES/FASCICULES'  # ← à adapter

# Dossier de sortie — nomme-le comme dans stats_pages.py si tu veux comparer direct :
#   ROOT/ocr/re_ocr/results/re_ocr_results_extract_<date>/  (voir ALTO_SOURCES['Pero'])
OUTPUT_DIR = '/content/gdrive/MyDrive/reocr_pero_results'

# Modèle PERO OCR : zip du model zoo (https://pero-ocr.fit.vutbr.cz/models)
# ex. un modèle "print" français ou multilingue européen — copie l'URL du .zip ici.
MODEL_ZIP_URL  = 'https://COLLE-ICI-URL-DU-MODELE.zip'  # ← à adapter
MODEL_DIR      = '/content/pero_model'                  # décompressé localement (rapide, pas sur Drive)
MODEL_CONFIG   = f'{MODEL_DIR}/config.ini'              # ajusté après décompression si besoin

# Résolution des images IIIF Gallica (si pas d'images locales sur Drive)
IIIF_WIDTH = 2000

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Source  : {SOURCE_DIR}')
print(f'Sortie  : {OUTPUT_DIR}')
print(f'Modèle  : {MODEL_CONFIG}')


In [ ]:
# ── Cellule 4 : Téléchargement / préparation du modèle PERO OCR ───────────
import os, zipfile, glob

os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(MODEL_CONFIG):
    zip_path = f'{MODEL_DIR}/model.zip'
    if not os.path.exists(zip_path):
        print('⬇️  Téléchargement du modèle…')
        os.system(f'wget -q -O "{zip_path}" "{MODEL_ZIP_URL}"')
    print('📦 Décompression…')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(MODEL_DIR)

    # Le zip du model zoo contient souvent un sous-dossier — on cherche le config.ini
    if not os.path.exists(MODEL_CONFIG):
        found = glob.glob(f'{MODEL_DIR}/**/config.ini', recursive=True)
        if found:
            MODEL_CONFIG = found[0]
            print(f'  config.ini trouvé : {MODEL_CONFIG}')
        else:
            raise FileNotFoundError(
                f"config.ini introuvable dans {MODEL_DIR} — vérifie MODEL_ZIP_URL "
                "et la structure du zip (liste : " + str(glob.glob(f'{MODEL_DIR}/**/*', recursive=True))[:500] + ')'
            )

print(f'✅ Modèle prêt : {MODEL_CONFIG}')


In [ ]:
# ── Cellule 5 : Initialisation du PageParser PERO OCR ─────────────────────
import configparser
from pero_ocr.document_ocr.page_parser import PageParser

config = configparser.ConfigParser()
config.read(MODEL_CONFIG)

page_parser = PageParser(config, config_path=os.path.dirname(MODEL_CONFIG))
print('PageParser PERO OCR prêt.')

# Sanity check : méthodes d'export disponibles sur PageLayout (varie selon version pero-ocr)
from pero_ocr.document_ocr.layout import PageLayout
_dummy_methods = [m for m in dir(PageLayout) if 'alto' in m.lower() or 'xml' in m.lower()]
print('Méthodes export dispo sur PageLayout :', _dummy_methods)


In [ ]:
# ── Cellule 6 : Fonctions utilitaires (ARK, IIIF, ALTO) ───────────────────
import re
import xml.etree.ElementTree as ET
from pathlib import Path

NS_ALTO = 'http://www.loc.gov/standards/alto/ns-v3#'

def get_ark_from_manifest(manifest_path: Path) -> str:
    """Extrait l'ARK Gallica depuis manifest.xml."""
    tree = ET.parse(manifest_path)
    root = tree.getroot()
    text = ET.tostring(root, encoding='unicode')
    m = re.search(r'ark:/12148/(bpt6k[\w]+)', text)
    return m.group(1) if m else None

def iiif_url(ark: str, page: int, width: int = 2000) -> str:
    """URL IIIF Gallica pour une page donnée."""
    return (f'https://gallica.bnf.fr/iiif/ark:/12148/{ark}'
            f'/f{page}/full/{width},/0/native.jpg')

def get_alto_page_num(alto_filename: str) -> int:
    """X0000001.xml → 1"""
    m = re.search(r'X0*(\d+)', alto_filename)
    return int(m.group(1)) if m else 1

def download_image(url: str, dest: Path) -> Path:
    """Télécharge une image IIIF si pas déjà en cache local."""
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        dest.write_bytes(resp.content)
    return dest

print('Fonctions utilitaires chargées.')


In [ ]:
# ── Cellule 7 : Traitement d'une page ──────────────────────────────────────
import cv2
import numpy as np
import requests

def run_pero_on_page(image_path: Path, page_id: str):
    """
    Charge une image et fait tourner PERO OCR dessus.
    Retourne l'objet PageLayout (contient régions, lignes, texte, coordonnées).
    """
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)  # BGR, comme attendu par pero-ocr
    if image is None:
        raise ValueError(f"Impossible de lire l'image : {image_path}")

    page_layout = PageLayout(id=page_id, page_size=(image.shape[0], image.shape[1]))
    page_layout = page_parser.process_page(image, page_layout)
    return page_layout


def save_alto(page_layout, out_path: Path):
    """Exporte le PageLayout en ALTO XML."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    # Le nom de méthode a varié selon les versions de pero-ocr (to_altoxml / export_alto).
    if hasattr(page_layout, 'to_altoxml'):
        page_layout.to_altoxml(str(out_path))
    elif hasattr(page_layout, 'export_alto'):
        page_layout.export_alto(str(out_path))
    else:
        raise AttributeError(
            "Aucune méthode d'export ALTO trouvée sur PageLayout — "
            "vérifie la version de pero-ocr installée (dir(page_layout))"
        )


def process_page(fascicule_id: str, alto_path: Path, out_ocr_dir: Path,
                 img_cache_dir: Path, ark: str | None, local_img_dir: Path | None):
    pnum = get_alto_page_num(alto_path.name)
    out_alto = out_ocr_dir / f'X{pnum:07d}.xml'
    if out_alto.exists():
        return {'status': 'cache', 'page': pnum}

    # 1. Image locale (si dispo, ex. Sample/pt1/pt2/pt3/<fasc>/master/T0000001.jp2) sinon IIIF Gallica
    image_path = None
    if local_img_dir:
        candidate = local_img_dir / f'T{pnum:07d}.jp2'
        if candidate.exists():
            image_path = candidate
    if image_path is None:
        if not ark:
            return {'status': 'skip_no_image', 'page': pnum}
        url = iiif_url(ark, pnum, IIIF_WIDTH)
        image_path = download_image(url, img_cache_dir / f'{fascicule_id}_p{pnum}.jpg')

    try:
        page_layout = run_pero_on_page(image_path, page_id=f'{fascicule_id}_X{pnum:07d}')
        save_alto(page_layout, out_alto)
        n_lines = sum(len(region.lines) for region in page_layout.regions)
        return {'status': 'ok', 'page': pnum, 'n_lines': n_lines}
    except Exception as e:
        print(f'    p{pnum} ❌ {e}')
        return {'status': 'error', 'page': pnum, 'erreur': str(e)}

print('Fonctions de traitement chargées.')


In [ ]:
# ── Cellule 8 : Test rapide sur une seule page ─────────────────────────────
# Vérifie que le modèle + le pipeline fonctionnent avant de lancer le lot complet.

_test_folders = sorted(Path(SOURCE_DIR).glob('*_reocr'))
if _test_folders:
    _f = _test_folders[0]
    _fid = re.sub(r'\D', '', _f.name)
    _manifest = _f / 'manifest.xml'
    _ark = get_ark_from_manifest(_manifest) if _manifest.exists() else None
    _alto_files = sorted((_f / 'ocr').glob('X*.xml'))
    if _alto_files and _ark:
        _test_url = iiif_url(_ark, get_alto_page_num(_alto_files[0].name), IIIF_WIDTH)
        _test_img = download_image(_test_url, Path('/content/test_page.jpg'))
        _layout = run_pero_on_page(_test_img, page_id='test')
        print(f'✅ Test OK — {sum(len(r.lines) for r in _layout.regions)} lignes détectées')
        print('Premières lignes :')
        for r in _layout.regions[:2]:
            for l in r.lines[:3]:
                print('   ', l.transcription)
    else:
        print('⚠ Pas de manifest/ARK exploitable pour le test — vérifie SOURCE_DIR')
else:
    print(f'⚠ Aucun dossier *_reocr trouvé dans {SOURCE_DIR}')


In [ ]:
# ── Cellule 9 : Pipeline principal ─────────────────────────────────────────
import shutil, time

source_path   = Path(SOURCE_DIR)
folders       = sorted(source_path.glob('*_reocr'))
img_cache_dir = Path('/content/iiif_cache')
print(f'{len(folders)} fascicules trouvés\n')

all_results = []

for folder in folders:
    fascicule_id = re.sub(r'\D', '', folder.name)
    print(f'\n══ {fascicule_id} ══')

    manifest = folder / 'manifest.xml'
    ark = get_ark_from_manifest(manifest) if manifest.exists() else None
    if not ark:
        print('  ⚠ manifest.xml / ARK absent — on tentera les images locales seulement')

    ocr_dir = folder / 'ocr'
    alto_files = sorted(ocr_dir.glob('X*.xml'))
    print(f'  {len(alto_files)} page(s) à traiter')

    out_dir     = Path(OUTPUT_DIR) / f'{fascicule_id}_reocr'
    out_ocr_dir = out_dir / 'ocr'
    out_ocr_dir.mkdir(parents=True, exist_ok=True)

    # Copier manifest.xml et toc/ — identique aux autres pipelines de re-OCR
    if manifest.exists():
        shutil.copy2(manifest, out_dir / 'manifest.xml')
    if (folder / 'toc').exists():
        shutil.copytree(folder / 'toc', out_dir / 'toc', dirs_exist_ok=True)

    for alto_path in alto_files:
        r = process_page(fascicule_id, alto_path, out_ocr_dir,
                         img_cache_dir, ark, local_img_dir=None)
        status_icon = {'ok': '✓', 'cache': '↩', 'skip_no_image': '⚠', 'error': '❌'}.get(r['status'], '?')
        print(f"  p{r['page']} {status_icon} {r['status']}"
              + (f" — {r['n_lines']} lignes" if r.get('n_lines') is not None else ''))
        all_results.append({**r, 'fascicule': fascicule_id})
        time.sleep(0.2)

print(f'\n✅ Terminé — {len(all_results)} page(s) traitées')


In [ ]:
# ── Cellule 10 : Rapport de synthèse ───────────────────────────────────────
import json
from pathlib import Path

ok     = [r for r in all_results if r['status'] == 'ok']
cached = [r for r in all_results if r['status'] == 'cache']
errors = [r for r in all_results if r['status'] == 'error']

rapport = {
    'total_pages': len(all_results),
    'total_fascicules': len({r['fascicule'] for r in all_results}),
    'ok': len(ok), 'cache': len(cached), 'erreurs': len(errors),
    'pages': all_results,
}
rapport_path = Path(OUTPUT_DIR) / 'rapport_pero_ocr.json'
rapport_path.write_text(json.dumps(rapport, ensure_ascii=False, indent=2))

print(f'Fascicules traités : {rapport["total_fascicules"]}')
print(f'Pages OK           : {rapport["ok"]}')
print(f'Pages en cache     : {rapport["cache"]}')
print(f'Erreurs            : {rapport["erreurs"]}')
print(f'Rapport            : {rapport_path}')
